# S23DR 2026 — HSS Evaluation

Evaluates a trained `RoofWireframeNet` checkpoint on the validation split
using the **Hausdorff Segment Score (HSS)**.

**Steps**
1. Check GPU
2. Mount Google Drive
3. Install dependencies & pull latest repo code
4. Load checkpoint from Drive
5. Load the S23DR validation dataset
6. Run inference + compute HSS
7. Results summary & confidence threshold sweep

## 1 · Check GPU

In [ ]:
import torch

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU      : {props.name}  ({props.total_memory / 1e9:.1f} GB)")
    DEVICE = "cuda"
else:
    print("GPU      : not available — using CPU (will be slow)")
    DEVICE = "cpu"

print(f"Device   : {DEVICE}")

## 2 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
CKPT_PATH = "/content/drive/MyDrive/s23dr_last.pt"   # ← change if needed
assert os.path.exists(CKPT_PATH), f"Checkpoint not found: {CKPT_PATH}"
print(f"Checkpoint found: {CKPT_PATH}  ({os.path.getsize(CKPT_PATH)/1e6:.1f} MB)")

## 3 · Install dependencies & pull latest code

In [ ]:
!pip install -q datasets huggingface_hub scipy numpy

In [ ]:
import os

REPO_DIR = "/content/3d_building_construction"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull origin main
else:
    !git clone https://github.com/12turtleships/3d_building_construction {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 4 · Load checkpoint

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import torch
from s23dr.model import RoofWireframeNet, WireframeLoss

device = torch.device(DEVICE)

try:
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
except TypeError:
    ckpt = torch.load(CKPT_PATH, map_location=device)

saved_args = ckpt.get("args", {})
N_QUERIES  = saved_args.get("n_queries", 64)
epoch      = ckpt.get("epoch", "?")
val_loss   = ckpt.get("val_loss", float("nan"))

print(f"epoch     = {epoch}")
print(f"val_loss  = {val_loss:.4f}")
print(f"n_queries = {N_QUERIES}")

model = RoofWireframeNet(n_queries=N_QUERIES).to(device)
model.load_state_dict(ckpt["model"])
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")
print("Model loaded successfully.")

## 5 · Load the S23DR validation dataset

Streams from the public HF dataset — no local download needed.  
If you have the dataset saved locally via `save_to_disk`, set `USE_LOCAL = True`.

In [ ]:
# ── configuration ────────────────────────────────────────────────────────────
USE_LOCAL  = False                              # True → load from LOCAL_DATA_DIR
LOCAL_DATA_DIR = "/content/3d_building_construction/s23dr/data"
HF_DATASET = "usm3d/s23dr-2026-sampled_4096_v2"
SPLIT      = "validation"
N_POINTS   = 1024
BATCH_SIZE = 8
CONF_THRESH = 0.5   # vertex confidence threshold (tuned in step 7)
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import io, zipfile
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


def _unpack_row(row):
    blob = row["data"]
    out = {}
    with zipfile.ZipFile(io.BytesIO(blob)) as zf:
        for name in zf.namelist():
            if name.endswith(".npy"):
                out[name[:-4]] = np.load(io.BytesIO(zf.read(name)), allow_pickle=False)
    out["order_id"] = row.get("order_id", "")
    return out


class S23DRValDataset(Dataset):
    def __init__(self, rows, n_points=1024):
        self._rows = [_unpack_row(r) for r in rows]
        self.n_points = n_points

    def __len__(self):
        return len(self._rows)

    def __getitem__(self, idx):
        r = self._rows[idx]
        N   = min(self.n_points, len(r["xyz_norm"]))
        sel = np.arange(N)
        return {
            "order_id": r["order_id"],
            "xyz":       torch.from_numpy(r["xyz_norm"][sel]).float(),
            "vote_frac": torch.from_numpy(r["vote_frac"][sel]).float(),
            "n_views":   torch.from_numpy(r["n_views_voted"][sel].astype(np.float32)).float(),
            "mask":      torch.from_numpy(r["mask"][sel].astype(np.float32)).float(),
            "class_id":  torch.from_numpy(r["class_id"][sel].astype(np.int64)),
            "gt_segs":   torch.from_numpy(r["gt_segments"]).float(),  # (E, 2, 3)
        }


def _collate(batch):
    return {
        "order_id":  [b["order_id"]  for b in batch],
        "xyz":       torch.stack([b["xyz"]       for b in batch]),
        "vote_frac": torch.stack([b["vote_frac"] for b in batch]),
        "n_views":   torch.stack([b["n_views"]   for b in batch]),
        "mask":      torch.stack([b["mask"]      for b in batch]),
        "class_id":  torch.stack([b["class_id"]  for b in batch]),
        "gt_segs":   [b["gt_segs"].numpy()        for b in batch],
    }


# Load rows
if USE_LOCAL:
    from datasets import load_from_disk
    hf_ds = load_from_disk(LOCAL_DATA_DIR)
    rows  = list(hf_ds[SPLIT] if SPLIT in hf_ds else hf_ds)
else:
    from datasets import load_dataset
    rows = list(load_dataset(HF_DATASET, split=SPLIT))

val_ds = S23DRValDataset(rows, n_points=N_POINTS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, collate_fn=_collate)

print(f"Loaded {len(val_ds)} validation samples  (batch_size={BATCH_SIZE})")

In [ ]:
# ── Coordinate space diagnostic ─────────────────────────────────────────────
# Prints min/max/mean of each array to confirm pred_pos and gt_segments
# live in the same coordinate space.  If they differ by orders of magnitude
# the model will never score >0 HSS regardless of training quality.

r0 = val_ds._rows[0]   # raw unpacked dict for sample 0

print("── Raw dataset arrays (sample 0) ──")
for key in ("xyz_norm", "gt_vertices", "gt_segments"):
    arr = r0.get(key)
    if arr is None:
        print(f"  {key}: NOT FOUND")
    else:
        print(f"  {key}: shape={arr.shape}  "
              f"min={arr.min():.4f}  max={arr.max():.4f}  mean={arr.mean():.4f}")

# Run a single forward pass to inspect pred_pos range
_batch0 = val_ds[0]
with torch.no_grad():
    _out0 = model(
        _batch0["xyz"].unsqueeze(0).to(device),
        _batch0["vote_frac"].unsqueeze(0).to(device),
        _batch0["n_views"].unsqueeze(0).to(device),
        _batch0["mask"].unsqueeze(0).to(device),
        _batch0["class_id"].unsqueeze(0).to(device),
    )
pred_pos0 = _out0["pred_pos"][0].cpu().numpy()
print(f"\n  pred_pos (model output): shape={pred_pos0.shape}  "
      f"min={pred_pos0.min():.4f}  max={pred_pos0.max():.4f}  mean={pred_pos0.mean():.4f}")

# Also check if scale/center metadata is available
for key in ("scale", "center", "xyz_center", "xyz_scale"):
    val = r0.get(key)
    if val is not None:
        print(f"\n  {key}: {val}")

print("\n── Interpretation ──")
print("  gt_segments and pred_pos should be in the same range.")
print("  If gt_segments range is >> 1, it is in world space (metres)")
print("  and pred_pos (normalized) will never match → HSS = 0.")
print("  Fix: convert gt_segments to normalized space using scale/center,")
print("  OR convert pred_pos to world space before calling hss().")


## 6 · Diagnose model output, then run inference & compute HSS

In [ ]:
# Sanity-check a single batch before running the full eval.
# If edge_no_edge_frac ≈ 1.0 the model collapsed due to class imbalance
# during training → retrain with the fixed WireframeLoss (no_edge_weight=0.05).

diag_batch = next(iter(val_loader))
with torch.no_grad():
    diag_out = model(
        diag_batch["xyz"].to(device),
        diag_batch["vote_frac"].to(device),
        diag_batch["n_views"].to(device),
        diag_batch["mask"].to(device),
        diag_batch["class_id"].to(device),
    )

# Confidence distribution
confs = torch.sigmoid(diag_out["pred_conf"][0]).cpu()
print("── Vertex confidence (first sample) ──")
print(f"  min={confs.min():.3f}  max={confs.max():.3f}  mean={confs.mean():.3f}")
print(f"  active (>0.5): {(confs > 0.5).sum().item()}  "
      f"active (>0.1): {(confs > 0.1).sum().item()}")

# Edge prediction distribution
edge_cls = diag_out["edge_logits"][0].argmax(dim=-1).cpu()  # (K, K)
no_edge_class = 10
frac_no_edge = (edge_cls == no_edge_class).float().mean().item()
print("\n── Edge predictions (first sample) ──")
print(f"  Fraction predicting 'no edge' (class {no_edge_class}): {frac_no_edge:.4f}")
print(f"  Unique predicted classes: {edge_cls.unique().tolist()}")

if frac_no_edge > 0.999:
    print("\n⚠ Model predicts 'no edge' for every pair.")
    print("  Cause: class imbalance during training (no_edge_weight was missing).")
    print("  Fix:   retrain using the updated WireframeLoss(no_edge_weight=0.05).")
else:
    print("\n✓ Model is predicting some edges — proceeding to full eval.")

In [ ]:
import time
from s23dr.metrics import hss, decode_to_segments

loss_fn = WireframeLoss()


@torch.no_grad()
def evaluate(model, loader, conf_thresh):
    model.eval()
    results = []
    t0 = time.time()

    for i, batch in enumerate(loader):
        out = model(
            batch["xyz"].to(device),
            batch["vote_frac"].to(device),
            batch["n_views"].to(device),
            batch["mask"].to(device),
            batch["class_id"].to(device),
        )

        for b in range(batch["xyz"].shape[0]):
            verts, edges, _ = loss_fn.decode(
                out["pred_pos"][b],
                out["pred_conf"][b],
                out["edge_logits"][b],
                conf_thresh=conf_thresh,
            )
            pred_segs = decode_to_segments(verts, edges)
            gt_segs   = batch["gt_segs"][b]
            scores    = hss(pred_segs, gt_segs)
            results.append({"order_id": batch["order_id"][b], **scores})

        if (i + 1) % 10 == 0:
            mean_hss = np.mean([r["hss"] for r in results])
            print(f"  [{len(results):4d}/{len(loader.dataset)}]  "
                  f"mean_hss={mean_hss:.4f}  ({time.time()-t0:.1f}s)")

    return results


print(f"Evaluating with conf_thresh={CONF_THRESH} …")
eval_results = evaluate(model, val_loader, CONF_THRESH)
print(f"Done — {len(eval_results)} samples evaluated")

## 7 · Results summary & confidence threshold sweep

In [ ]:
import numpy as np

hss_scores  = np.array([r["hss"]       for r in eval_results])
prec_scores = np.array([r["precision"] for r in eval_results])
rec_scores  = np.array([r["recall"]    for r in eval_results])

print("=" * 45)
print(f"  conf_thresh : {CONF_THRESH}")
print(f"  Samples     : {len(hss_scores)}")
print("-" * 45)
print(f"  HSS         : {hss_scores.mean():.4f}  (std {hss_scores.std():.4f})")
print(f"  Precision   : {prec_scores.mean():.4f}")
print(f"  Recall      : {rec_scores.mean():.4f}")
print("=" * 45)

In [ ]:
# Sweep confidence thresholds to find the best HSS
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
sweep_results = []

print(f"{'thresh':>8}  {'HSS':>8}  {'Prec':>8}  {'Recall':>8}")
print("-" * 40)

for thresh in thresholds:
    res = evaluate(model, val_loader, conf_thresh=thresh)
    h = np.mean([r["hss"]       for r in res])
    p = np.mean([r["precision"] for r in res])
    rc = np.mean([r["recall"]   for r in res])
    sweep_results.append((thresh, h, p, rc))
    print(f"{thresh:>8.2f}  {h:>8.4f}  {p:>8.4f}  {rc:>8.4f}")

best = max(sweep_results, key=lambda x: x[1])
print("-" * 40)
print(f"Best conf_thresh = {best[0]}  →  HSS = {best[1]:.4f}")

In [ ]:
# Inspect the 10 worst-scoring samples for error analysis
sorted_results = sorted(eval_results, key=lambda r: r["hss"])

print("10 worst samples:")
print(f"{'order_id':>30}  {'HSS':>8}  {'Prec':>8}  {'Recall':>8}")
print("-" * 60)
for r in sorted_results[:10]:
    print(f"{str(r['order_id']):>30}  {r['hss']:>8.4f}  "
          f"{r['precision']:>8.4f}  {r['recall']:>8.4f}")

In [ ]:
# Optional: save per-sample results to Drive
import json, os

out_path = "/content/drive/MyDrive/s23dr_eval_results.json"
with open(out_path, "w") as f:
    json.dump({
        "checkpoint": CKPT_PATH,
        "split": SPLIT,
        "conf_thresh": CONF_THRESH,
        "mean_hss":       float(hss_scores.mean()),
        "mean_precision": float(prec_scores.mean()),
        "mean_recall":    float(rec_scores.mean()),
        "per_sample": eval_results,
    }, f, indent=2)

print(f"Results saved to {out_path}")

In [ ]:
# ── 8 · Visualise point cloud + predicted vs GT wireframe ────────────────────
# Interactive 3-D plotly chart for a single validation sample.
#
# Blue dots   = input point cloud (xyz_norm)
# Green lines = GT roof wireframe (gt_segments, normalised space)
# Red lines   = Predicted roof wireframe (at VIS_THRESH)
# Red diamonds= Predicted vertices

import plotly.graph_objects as go

SAMPLE_IDX = 0      # ← change to inspect a different sample
VIS_THRESH = 0.10   # confidence threshold (best from the sweep above)
MAX_PTS    = 2048   # subsample point cloud for faster rendering

# ── raw data ──────────────────────────────────────────────────────────────────
r        = val_ds._rows[SAMPLE_IDX]
xyz_np   = r["xyz_norm"]        # (N, 3)
gt_segs  = r["gt_segments"]     # (E, 2, 3)  normalised space

# ── inference ─────────────────────────────────────────────────────────────────
item = val_ds[SAMPLE_IDX]
with torch.no_grad():
    out = model(
        item["xyz"].unsqueeze(0).to(device),
        item["vote_frac"].unsqueeze(0).to(device),
        item["n_views"].unsqueeze(0).to(device),
        item["mask"].unsqueeze(0).to(device),
        item["class_id"].unsqueeze(0).to(device),
    )
verts, edges, _ = loss_fn.decode(
    out["pred_pos"][0], out["pred_conf"][0], out["edge_logits"][0],
    conf_thresh=VIS_THRESH,
)
pred_segs = decode_to_segments(verts, edges)   # (E_pred, 2, 3)

# ── point cloud trace ─────────────────────────────────────────────────────────
rng = np.random.default_rng(0)
vis_idx = rng.choice(len(xyz_np), min(MAX_PTS, len(xyz_np)), replace=False)
pc = xyz_np[vis_idx]

pc_trace = go.Scatter3d(
    x=pc[:, 0], y=pc[:, 1], z=pc[:, 2],
    mode="markers",
    marker=dict(size=1.5, color="royalblue", opacity=0.35),
    name="Point cloud",
)

# ── helper: (E, 2, 3) → NaN-separated line trace ─────────────────────────────
def _line_trace(segs, color, name, width=4):
    if len(segs) == 0:
        return go.Scatter3d(x=[], y=[], z=[], mode="lines",
                            line=dict(color=color, width=width), name=name)
    xs, ys, zs = [], [], []
    for s in segs:
        xs += [float(s[0, 0]), float(s[1, 0]), None]
        ys += [float(s[0, 1]), float(s[1, 1]), None]
        zs += [float(s[0, 2]), float(s[1, 2]), None]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode="lines",
                        line=dict(color=color, width=width), name=name)

gt_trace   = _line_trace(gt_segs,   color="limegreen", name=f"GT ({len(gt_segs)} segs)")
pred_trace = _line_trace(pred_segs, color="red",        name=f"Pred ({len(pred_segs)} segs)")

# ── predicted vertex markers ──────────────────────────────────────────────────
verts_np = verts.cpu().numpy() if hasattr(verts, "cpu") else np.array(verts)
vert_trace = go.Scatter3d(
    x=verts_np[:, 0] if len(verts_np) else [],
    y=verts_np[:, 1] if len(verts_np) else [],
    z=verts_np[:, 2] if len(verts_np) else [],
    mode="markers",
    marker=dict(size=6, color="red", symbol="diamond"),
    name=f"Pred vertices ({len(verts_np)})",
)

hss_score = hss(pred_segs, gt_segs)

fig = go.Figure(data=[pc_trace, gt_trace, pred_trace, vert_trace])
fig.update_layout(
    title=(f"Sample {SAMPLE_IDX} | {r['order_id']} | thresh={VIS_THRESH} | "
           f"HSS={hss_score['hss']:.3f}  P={hss_score['precision']:.3f}  R={hss_score['recall']:.3f}"),
    scene=dict(aspectmode="data",
               xaxis_title="X (norm)", yaxis_title="Y (norm)", zaxis_title="Z (norm)"),
    legend=dict(x=0, y=1),
    margin=dict(l=0, r=0, t=50, b=0),
    height=680,
)
fig.show()

print(f"GT segments   : {len(gt_segs)}")
print(f"Pred segments : {len(pred_segs)}")
print(f"Pred vertices : {len(verts_np)}")
print(f"HSS={hss_score['hss']:.4f}  Precision={hss_score['precision']:.4f}  Recall={hss_score['recall']:.4f}")


## 9 · Dataset schema: all arrays inside each ZIP sample

Each row in `usm3d/s23dr-2026-sampled_4096_v2` (and v3) stores a ZIP
archive in the `data` column. This cell unpacks one sample and prints
every `.npy` array with its shape, dtype, and value range.

**Confirmed arrays** (from `s23dr/data.py` and live inspection):

| Array | Shape | Dtype | Description |
|-------|-------|-------|-------------|
| `xyz_norm` | (N, 3) | float32 | Normalised 3-D point positions. World space = `xyz_norm * scale + center`. |
| `vote_frac` | (N,) | float32 | Per-point view-agreement score (0–1): fraction of SfM camera views that agree on the 3-D position. High values = reliable points. |
| `n_views_voted` | (N,) | int/float | Number of SfM views that contributed to each point. Normalised by `/8` before feeding to the model. |
| `mask` | (N,) | bool/float | Valid-point indicator. Some SfM points may be filtered; mask=0 suppresses them during training. |
| `class_id` | (N,) | int64 | Per-point semantic class label (0–63). Embedded via a learnable 8-D lookup table in the model. |
| `gt_vertices` | (V, 3) | float32 | Ground-truth wireframe vertices in **world space** (metres). Convert to normalised space: `(gt_vertices - center) / scale`. |
| `gt_edges` | (E, 2) | int64 | Edge connectivity: pairs of indices into `gt_vertices`. |
| `gt_edge_classes` | (E,) | int64 | Semantic edge type (0–9): ridge, eave, hip, valley, wall, etc. |
| `gt_segments` | (E, 2, 3) | float32 | Pre-computed segment form of edges, already in **normalised** space. Used directly by the HSS metric. |
| `scale` | scalar | float32 | Scale factor: world = normalised × scale. |
| `center` | (3,) | float32 | Translation: world = normalised × scale + center. |

| `source` | (N,) | uint8 | Target-building mask (1 = belongs to the house being predicted, 0 = neighbouring structure or background). Filter to `source==1` before running the pipeline. |
| `visible_src` | (N,) | uint8 | Which camera source contributed the point (1 or 2). |
| `visible_id` | (N,) | int16 | Camera view ID that originally observed the point. |
| `behind` | (N,) | int16 | Number of other surfaces behind this point (depth layer index). |

> **Note**: the `xyz_norm` array in the **test split** (what you submit against) contains
> `xyz_norm`, `vote_frac`, `n_views_voted`, `mask`, and `class_id` but **no GT arrays**.
> GT arrays only exist in the `train` and `validation` splits.


In [ ]:
# Inspect all numpy arrays in the first validation sample
import io, zipfile
import numpy as np

raw_row = rows[0]   # first sample (rows loaded in step 5)
blob = raw_row["data"]

print(f"ZIP byte length : {len(blob):,}")
print(f"order_id        : {raw_row['order_id']}")
print()
print(f"{'Array':25s}  {'Shape':18s}  {'Dtype':10s}  {'Min':>10s}  {'Max':>10s}  {'Mean':>10s}")
print("-" * 90)

with zipfile.ZipFile(io.BytesIO(blob)) as zf:
    for name in sorted(zf.namelist()):
        arr = np.load(io.BytesIO(zf.read(name)), allow_pickle=False)
        key = name.replace(".npy", "")
        mn  = float(arr.min())  if arr.size > 0 else float("nan")
        mx  = float(arr.max())  if arr.size > 0 else float("nan")
        mu  = float(arr.mean()) if arr.size > 0 else float("nan")
        print(f"{key:25s}  {str(arr.shape):18s}  {str(arr.dtype):10s}  "
              f"{mn:>10.4f}  {mx:>10.4f}  {mu:>10.4f}")


## 10 · S23DR 2026 — Competition Study

### Overview

| Item | Detail |
|------|--------|
| Competition | **Structured 3D Reconstruction 2026 (S23DR 2026)** |
| Workshop | USM3D @ CVPR 2026 |
| Prize | $12,000 |
| Task | Predict 3D roof wireframe from a sparse SfM point cloud |
| Metric | **HSS** (Hausdorff Segment Score) — F1 of precision & recall at segment Hausdorff distance τ=0.2 (normalised space) |
| Space | `usm3d/S23DR2026` — Docker, running on `cpu-upgrade` hardware |
| Submission | `submission.parquet` with columns `__key__`, `wf_vertices`, `wf_edges` |

---

### HSS Metric (arXiv 2503.08208)

```
precision = |{p ∈ pred : min Hausdorff(p, GT) < τ}| / |pred|
recall    = |{g ∈ GT   : min Hausdorff(g, pred) < τ}| / |GT|
HSS       = 2 · precision · recall / (precision + recall)
```

Hausdorff is computed by sampling 8 points along each segment.
τ = **0.2** in normalised space (≈ 20% of the bounding box diagonal).
This is stricter than it looks — a typical house has a 4-segment roof so
each miss costs 25% recall.

---

### Datasets

| Dataset | Rows (train / val) | Size | Notes |
|---------|--------------------|------|-------|
| `usm3d/s23dr-2026-sampled_4096_v2` | 15.9K / 1.0K | 1.1 GB | Baseline training set (4096 pts, **v2**) |
| `usm3d/s23dr-2026-sampled_4096_v3` | 17.5K / 170 | 1.2 GB | Updated version with more samples |
| `usm3d/s23dr-2026-sampled_2048_v3` | 17.5K / 170 | 619 MB | Sparser input for faster training |
| `usm3d/s23dr-2026-sampled_8192_v3` | 17.5K / 170 | ~2.5 GB | Denser input for higher fidelity |
| `usm3d/s23dr-2026-cached_full_pcd_v2` | 16.5K / 1.0K | 16.1 GB | Full unsampled point clouds |
| `usm3d/hoho22k_2026_trainval` | ~20.7K | — | **Gated.** Multi-modal: images, depth, ADE20K segmentation, gestalt, camera K/R/t, GT wireframes |

**Recommendation**: train on `sampled_4096_v3` (most downloads, newest).
Use `cached_full_pcd_v2` for a high-fidelity run once the pipeline is working.

---

### Official Baselines (`usm3d` org)

| Model | Description |
|-------|-------------|
| `usm3d/empty_submission_2026` | All-empty wireframes (HSS = 0, score floor) |
| `usm3d/handcrafted_submission_2026` | Rule-based geometric approach |
| `usm3d/learned-baseline-2026` | The PointNet-style model in **this repo** (arXiv 2503.08208, CC BY-NC 4.0) |

The learned baseline uses the same architecture as `RoofWireframeNet`:
PointNet backbone → cross-attention vertex decoder → pairwise edge predictor.

---

### SOTA / Competitive Methods

| Method | Source | Key idea | HSS (S23DR) |
|--------|--------|----------|-------------|
| **Procedural (this repo)** | `s23dr/procedural/` | Normal segmentation → RANSAC planes → alpha-shape eaves → graph pruning; no training needed | **0.5052** (val, 1024 samples) |
| Two-stage PointNet (2025 winner) | — | Vertex refinement + cylindrical-patch edge prediction | 0.43 |
| Delaunay Canopy | arXiv 2604.02497 (Apr 2026) | Delaunay graph prior + curvature-guided reconstruction; SOTA on Building3D | not published yet |
| 3D House Wireframe w/ Semantics | arXiv 2407.12267 | Autoregressive graph-encoder + transformer decoder | — |
| Point2Building | arXiv 2403.02136 | Autoregressive 3D polygon mesh from airborne LiDAR | — |

**Procedural precision/recall breakdown**: P=0.490 R=0.702 — pipeline recovers
most GT edges (high recall) but also emits ~50% spurious edges (low precision).
Tightening edge-selection thresholds or adding a learned edge scorer is the
highest-priority improvement.

---

### Current Architecture (this repo)

```
Input: xyz_norm (N,3) + vote_frac (N,) + n_views (N,) + mask (N,) + class_id (N,)
  └─> class_embed: Embedding(64, 8)
  └─> concat → (N, 14)

PointNetBackbone
  per-point MLP: [14 → 64 → 128 → 256]
  global MLP:    [256 → 512 → 1024]  + max-pool → global_feat (1024,)

VertexDecoder
  K=64 learned query embeddings (256-D)
  cross-attention over per-point features (256-D)
  attention-weighted position anchor + MLP offset → pred_pos  (K, 3)
  confidence head → pred_conf  (K,)

EdgePredictor
  pair feature = feat_i || feat_j || global || dist || diff  (dim ~1285)
  MLP → 11 classes (10 edge types + no-edge)
  → edge_logits  (K, K, 11)

Loss
  Hungarian matching (linear_sum_assignment) for vertex assignment
  L1 vertex position + BCE confidence + weighted cross-entropy edge
  no_edge_weight=0.05 to handle the ~100:1 negative/positive edge imbalance
```

---

### Why `xyz_norm` shows vertical streaks (investigation result)

The dense collinear point clusters visible as streaks in 3-D scatter plots
are **real SfM structure**, not a rendering artefact:

1. `mode="markers"` is set correctly — not a Plotly line-mode issue.
2. No NaN values in the array — not an edge-list format artefact.
3. Point norms ≈ 0.33 — not surface normals (which would be ≈ 1.0).

**Root cause**: SfM naturally places dense clusters of collinear points on
architectural edge features (window frames, building corners, wall junctions)
because those are the most reliably matched features across camera views.
Dozens of points millimetres apart on the same vertical edge render as a
solid streak at marker size 1.5.

---

### Improvement roadmap

Listed roughly by expected HSS gain / effort ratio:

1. **Train longer / bigger**: increase `n_queries` (64→128), use `n_points=4096` (full cloud),
   add cosine LR warm-up, train for 200+ epochs on `sampled_4096_v3`.

2. **Upgrade backbone**: replace PointNet with
   [PointNext](https://arxiv.org/abs/2206.04670) or
   [Point Transformer v3](https://arxiv.org/abs/2312.10035) for richer local geometry.

3. **Delaunay candidate graph**: build a Delaunay graph from the point cloud and
   use it as an edge prior (see arXiv 2604.02497). Edges between Delaunay
   neighbours are far more likely to be real roof edges.

4. **Cylindrical-patch edge prediction** (2025 winner approach): for each
   candidate vertex pair, extract a local point-cloud cylinder and predict edge
   existence from the local geometry — avoids the O(K²) all-pairs bottleneck.

5. **Use `vote_frac` and `class_id` more aggressively**: current model embeds
   them but a higher-capacity backbone (e.g. with skip connections) may extract
   more signal from the semantic labels.

6. **Data augmentation**: random rotation about the vertical axis, random
   scale jitter, random point dropout beyond the existing subsample.

7. **Ensemble**: run inference at 2048 / 4096 / 8192 points and merge the
   three predicted wireframes (NMS on vertices + union of edges).

8. **Full point cloud**: fine-tune on `cached_full_pcd_v2` (16 GB) for the
   final submission — richer input should help on complex geometries.
